In [1]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt

In [2]:
df = pd.read_csv('../data/course_lead_scoring_2026.csv')

In [3]:
df

,lead_source,industry,employment_status,location,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
0,organic_search,technology,employed,europe,88160.0,3,4,0.64,1
1,social_media,technology,employed,south_america,72688.0,3,7,0.72,1
2,referral,retail,employed,europe,44697.0,3,4,0.58,1
3,paid_ads,manufacturing,student,NaN,NaN,3,6,0.57,0
4,referral,manufacturing,student,north_america,24062.0,4,5,0.62,1
...,...,...,...,...,...,...,...,...,...
4995,organic_search,education,employed,asia,45834.0,2,2,0.37,1
4996,NaN,technology,employed,north_america,84831.0,2,5,0.51,0
4997,organic_search,finance,employed,asia,53636.0,4,6,0.75,1
4998,referral,technology,employed,africa,69430.0,4,8,0.79,1


In [4]:
df.isnull().sum()

lead_source                 148
industry                    240
employment_status           194
location                    208
annual_income               369
number_of_courses_viewed      0
interaction_count             0
lead_score                   35
converted                     0
dtype: int64

In [5]:
df.dtypes

lead_source                     str
industry                        str
employment_status               str
location                        str
annual_income               float64
number_of_courses_viewed      int64
interaction_count             int64
lead_score                  float64
converted                     int64
dtype: object

In [6]:
categorical = list(df.dtypes[df.dtypes == 'str'].index)
categorical

['lead_source', 'industry', 'employment_status', 'location']

In [7]:
numerical = list(df.dtypes[(df.dtypes == 'int') | (df.dtypes == 'float')].index)
numerical

['annual_income',
 'number_of_courses_viewed',
 'interaction_count',
 'lead_score',
 'converted']

In [8]:
df[categorical] = df[categorical].fillna('NA')

In [9]:
df[numerical] = df[numerical].fillna(0.0)

In [10]:
df.isnull().sum()

lead_source                 0
industry                    0
employment_status           0
location                    0
annual_income               0
number_of_courses_viewed    0
interaction_count           0
lead_score                  0
converted                   0
dtype: int64

### Q1

In [11]:
df.industry.value_counts()

industry
technology       1173
retail            925
healthcare        840
finance           720
education         639
manufacturing     463
NA                240
Name: count, dtype: int64

In [12]:
# technology

### Q2

In [13]:
df[numerical].corr()

,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
annual_income,1.000000,0.161300,0.122842,0.229496,0.189652
number_of_courses_viewed,0.161300,1.000000,0.721609,0.757204,0.395038
interaction_count,0.122842,0.721609,1.000000,0.915746,0.448624
lead_score,0.229496,0.757204,0.915746,1.000000,0.483246
converted,0.189652,0.395038,0.448624,0.483246,1.000000


In [14]:
df[numerical].corr().stack().sort_values(ascending=False).reset_index()

,level_0,level_1,0
0,annual_income,annual_income,1.000000
1,number_of_courses_viewed,number_of_courses_viewed,1.000000
2,converted,converted,1.000000
3,lead_score,lead_score,1.000000
4,interaction_count,interaction_count,1.000000
5,interaction_count,lead_score,0.915746
6,lead_score,interaction_count,0.915746
7,lead_score,number_of_courses_viewed,0.757204
8,number_of_courses_viewed,lead_score,0.757204
9,interaction_count,number_of_courses_viewed,0.721609


In [15]:
# interaction_count AND lead_score

### Q3

In [16]:
len(df)

5000

In [17]:
from sklearn.model_selection import train_test_split

In [18]:
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=42)
df_train, df_val = train_test_split(df_full_train, test_size=0.25, random_state=42)

In [19]:
len(df_train), len(df_val), len(df_test)

(3000, 1000, 1000)

In [20]:
df_train = df_train.reset_index(drop=True)

In [21]:
df_val = df_val.reset_index(drop=True)

In [22]:
df_test = df_test.reset_index(drop=True)

In [23]:
y_train = df_train.converted
y_val = df_val.converted
y_test = df_test.converted

In [24]:
del df_train['converted']
del df_val['converted']
del df_test['converted']

In [25]:
from sklearn.metrics import mutual_info_score

def mutual_info_churn_score(series):
    return mutual_info_score(series, y_train)

In [26]:
mi = df_train[categorical].apply(mutual_info_churn_score)

In [27]:
mi = mi.sort_values(ascending=False).round(2)
mi

lead_source          0.03
employment_status    0.02
industry             0.00
location             0.00
dtype: float64

In [28]:
# lead_source

### Q4

In [29]:
dicts = df[numerical + categorical].to_dict(orient='records')

In [31]:
numerical.remove('converted')

In [32]:
from sklearn.feature_extraction import DictVectorizer

dv = DictVectorizer(sparse=False)

train_dict = df_train[numerical + categorical].to_dict(orient='records')
X_train = dv.fit_transform(train_dict)

In [33]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(
    solver='liblinear',
    C=1.0,
    max_iter=1000,
    random_state=42
)

model.fit(X_train, y_train)

,"random_state random_state: int, RandomState instance, default=NoneOnly used for `solver` == 'sag', 'saga' or 'liblinear' to shuffle thedata. It has no effect on the other solvers.See :term:`Glossary <random_state>` for details.",42
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:`multiclass` problems (`n_classes >= 3`), all solvers except 'liblinear' minimize the full multinomial loss, 'liblinear' will raise an error.- 'newton-cholesky' is a good choice for `n_samples` >> `n_features * n_classes`, especially with one-hot encoded categorical features with rare categories. Be aware that the memory usage of this solver has a quadratic dependency on `n_features * n_classes` because it explicitly computes the full Hessian matrix.- For small datasets, 'liblinear' is a good choice, whereas 'sag' and 'saga' are faster for large ones;- 'liblinear' can only handle binary classification by default. To apply a one-versus-rest scheme for the multiclass setting one can wrap it with the :class:`~sklearn.multiclass.OneVsRestClassifier`... warning:: The choice of the algorithm depends on the penalty chosen (`l1_ratio=0` for L2-penalty, `l1_ratio=1` for L1-penalty and `0 < l1_ratio < 1` for Elastic-Net) and on (multinomial) multiclass support: ================= ======================== ====================== solver l1_ratio multinomial multiclass ================= ======================== ====================== 'lbfgs' l1_ratio=0 yes 'liblinear' l1_ratio=1 or l1_ratio=0 no 'newton-cg' l1_ratio=0 yes 'newton-cholesky' l1_ratio=0 yes 'sag' l1_ratio=0 yes 'saga' 0<=l1_ratio<=1 yes ================= ======================== ======================.. note:: 'sag' and 'saga' fast convergence is only guaranteed on features with approximately the same scale. You can preprocess the data with a scaler from :mod:`sklearn.preprocessing`... seealso:: Refer to the :ref:`User Guide <Logistic_regression>` for more information regarding :class:`LogisticRegression` and more specifically the :ref:`Table <logistic_regression_solvers>` summarizing solver/penalty supports... versionadded:: 0.17 Stochastic Average Gradient (SAG) descent solver. Multinomial support in version 0.18... versionadded:: 0.19 SAGA solver... versionchanged:: 0.22 The default solver changed from 'liblinear' to 'lbfgs' in 0.22... versionadded:: 1.2 newton-cholesky solver. Multinomial support in version 1.6.",'liblinear'
,"max_iter max_iter: int, default=100Maximum number of iterations taken for the solvers to converge.",1000
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l

In [34]:
val_dict = df_val[numerical + categorical].to_dict(orient='records')
X_val = dv.transform(val_dict)

In [35]:
y_pred = model.predict_proba(X_val)[:, 1]

In [36]:
converted_decision = (y_pred >= 0.5)

In [38]:
(y_val == converted_decision).mean()

np.float64(0.645)

In [39]:
(y_val == converted_decision).mean().round(2)

np.float64(0.64)

In [40]:
# 0.65

### Q5

In [41]:
original_accuracy = (y_val == converted_decision).mean()
original_accuracy

np.float64(0.645)

In [42]:
features = numerical + categorical
features

['annual_income',
 'number_of_courses_viewed',
 'interaction_count',
 'lead_score',
 'lead_source',
 'industry',
 'employment_status',
 'location']

In [43]:
results = {}

for excluded in features:
    selected = [f for f in features if f != excluded]

    dv = DictVectorizer(sparse=False)

    train_dict = df_train[selected].to_dict(orient='records')
    X_train = dv.fit_transform(train_dict)

    val_dict = df_val[selected].to_dict(orient='records')
    X_val = dv.transform(val_dict)

    model = LogisticRegression(
        solver='liblinear',
        C=1.0,
        max_iter=1000,
        random_state=42
    )
    
    model.fit(X_train, y_train)

    y_pred = model.predict_proba(X_val)[:, 1]
    converted_decision = (y_pred >= 0.5)

    accuracy = (y_val == converted_decision).mean()

    results[excluded] = original_accuracy - accuracy

In [44]:
results

{'annual_income': np.float64(-0.07899999999999996),
 'number_of_courses_viewed': np.float64(0.0020000000000000018),
 'interaction_count': np.float64(0.04400000000000004),
 'lead_score': np.float64(0.0010000000000000009),
 'lead_source': np.float64(0.0030000000000000027),
 'industry': np.float64(0.0),
 'employment_status': np.float64(0.0),
 'location': np.float64(0.0)}

In [45]:
pd.Series(results).sort_values(ascending=False)

interaction_count           0.044
lead_source                 0.003
number_of_courses_viewed    0.002
lead_score                  0.001
employment_status           0.000
industry                    0.000
location                    0.000
annual_income              -0.079
dtype: float64

In [46]:
# lead_source                  0.003
# number_of_courses_viewed     0.002
# interaction_count            0.044


In [47]:
# number_of_courses_viewed

### Q6

In [48]:
numerical

['annual_income',
 'number_of_courses_viewed',
 'interaction_count',
 'lead_score']

In [49]:
categorical

['lead_source', 'industry', 'employment_status', 'location']

In [50]:
dv = DictVectorizer(sparse=False)

train_dict = df_train[numerical + categorical].to_dict(orient='records')
X_train = dv.fit_transform(train_dict)

val_dict = df_val[numerical + categorical].to_dict(orient='records')
X_val = dv.transform(val_dict)

scores = {}

for c in [0.000001, 0.00001, 0.0001, 0.001]:
    model = LogisticRegression(
        solver='liblinear',
        C=c,
        max_iter=1000,
        random_state=42
    )
    
    model.fit(X_train, y_train)

    y_pred = model.predict_proba(X_val)[:, 1]
    converted_decision = (y_pred >= 0.5)

    accuracy = (y_val == converted_decision).mean()

    scores[c] = round(accuracy, 3)

pd.Series(scores).sort_values(ascending=False)

0.001000    0.645
0.000100    0.613
0.000010    0.598
0.000001    0.598
dtype: float64

In [51]:
# 0.001